In [1]:
import pandas as pd
import numpy as np

## train: profile of borrower borrow at this time

In [46]:

train = pd.read_csv('../data/application_train.csv')
train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
train.describe()

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
count,307511.000000,307511.000000,307511.000000,3.075110e+05,3.075110e+05,307499.000000,3.072330e+05,307511.000000,307511.000000,307511.000000,...,307511.000000,307511.000000,307511.000000,307511.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000
mean,278180.518577,0.080729,0.417052,1.687979e+05,5.990260e+05,27108.573909,5.383962e+05,0.020868,-16036.995067,63815.045904,...,0.008130,0.000595,0.000507,0.000335,0.006402,0.007000,0.034362,0.267395,0.265474,1.899974
std,102790.175348,0.272419,0.722121,2.371231e+05,4.024908e+05,14493.737315,3.694465e+05,0.013831,4363.988632,141275.766519,...,0.089798,0.024387,0.022518,0.018299,0.083849,0.110757,0.204685,0.916002,0.794056,1.869295
min,100002.000000,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.000290,-25229.000000,-17912.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,189145.500000,0.000000,0.000000,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,0.010006,-19682.000000,-2760.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,278202.000000,0.000000,0.000000,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,0.018850,-15750.000000,-1213.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,367142.500000,0.000000,1.000000,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,0.028663,-12413.000000,-289.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
max,456255.000000,1.000000,19.000000,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,0.072508,-7489.000000,365243.000000,...,1.000000,1.000000,1.000000,1.000000,4.000000,9.000000,8.000000,27.000000,261.000000,25.000000


In [6]:
train.isnull().mean()

SK_ID_CURR                    0.000000
TARGET                        0.000000
NAME_CONTRACT_TYPE            0.000000
CODE_GENDER                   0.000000
FLAG_OWN_CAR                  0.000000
                                ...   
AMT_REQ_CREDIT_BUREAU_DAY     0.135016
AMT_REQ_CREDIT_BUREAU_WEEK    0.135016
AMT_REQ_CREDIT_BUREAU_MON     0.135016
AMT_REQ_CREDIT_BUREAU_QRT     0.135016
AMT_REQ_CREDIT_BUREAU_YEAR    0.135016
Length: 122, dtype: float64

In [7]:
train['TARGET'].value_counts(normalize=True)


TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [ ]:
def bad_rate_table_categorical(df, col, target='TARGET'):

    result = []


    table = (
            df.groupby(col)[target]
              .agg(Total='count', Bad='sum')
              .reset_index()
        )

    table['Good'] = table['Total'] - table['Bad']
    table['Bad Rate'] = (table['Bad'] / table['Total'] * 100).round(2)
    table["Population %"] = ((table['Total'] / table['Total'].sum())*100).round(2)
    table = table.rename(columns={col: 'Category'})

    result.append(table)

    return pd.concat(result, ignore_index=True)

In [ ]:
def bad_rate_table_numeric(df, col, target='TARGET', q = 5):
    temp = df[[col,target]].copy()
    temp['bins'] = pd.qcut(temp[col], q = q, duplicates='drop')
    
    table = (temp.groupby('bins', observed = False)[target].agg(
        Total = 'count',
        Bad = 'sum'
    ).reset_index())
    table['Population %'] = (table['Total'] / table['Total'].sum() * 100).round(2)
    table['Bad Rate'] = (table['Bad'] / table['Total'] * 100).round(2)

    return table[['Bin', 'Population %', 'Bad Rate']]

In [47]:
def train_application_features(df):

    app = df.copy()

    # =====================================================
    # Employment placeholder
    # 365243 = placeholder value
    # =====================================================
    app["EMPLOYED_PLACEHOLDER_FLAG"] = (
        app["DAYS_EMPLOYED"] == 365243
    ).astype(int)

    app.loc[
        app["DAYS_EMPLOYED"] == 365243,
        "DAYS_EMPLOYED"
    ] = np.nan

    # =====================================================
    # Income & Debt Capacity
    # =====================================================
    app["annuity_income_ratio"] = np.where(
        app["AMT_INCOME_TOTAL"] > 0,
        app["AMT_ANNUITY"] / app["AMT_INCOME_TOTAL"],
        np.nan
    )

    app["credit_income_ratio"] = np.where(
        app["AMT_INCOME_TOTAL"] > 0,
        app["AMT_CREDIT"] / app["AMT_INCOME_TOTAL"],
        np.nan
    )

    app["credit_goods_ratio"] = np.where(
        app["AMT_GOODS_PRICE"] > 0,
        app["AMT_CREDIT"] / app["AMT_GOODS_PRICE"],
        np.nan
    )

    app["income_per_family"] = np.where(
        app["CNT_FAM_MEMBERS"] > 0,
        app["AMT_INCOME_TOTAL"] / app["CNT_FAM_MEMBERS"],
        np.nan
    )

    # =====================================================
    # Age & Employment
    # =====================================================
    app["age_years"] = (
        -app["DAYS_BIRTH"] / 365
    )

    app["employment_years"] = (
        -app["DAYS_EMPLOYED"] / 365
    )

    app["employment_age_ratio"] = np.where(
        app["DAYS_BIRTH"] < 0,
        app["DAYS_EMPLOYED"] / app["DAYS_BIRTH"],
        np.nan
    )

    # =====================================================
    # External Scores
    # =====================================================
    ext_cols = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]

    app["ext_mean"] = app[ext_cols].mean(axis=1)

    app["ext_min"] = app[ext_cols].min(axis=1)

    app["ext_max"] = app[ext_cols].max(axis=1)

    app["ext_std"] = app[ext_cols].std(axis=1)

    app["ext_missing_count"] = (
        app[ext_cols]
        .isna()
        .sum(axis=1)
    )

    return app

In [48]:
train_feature = train_application_features(train)
train_feature.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,credit_goods_ratio,income_per_family,age_years,employment_years,employment_age_ratio,ext_mean,ext_min,ext_max,ext_std,ext_missing_count
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,1.158397,202500.0,25.920548,1.745205,0.067329,0.161787,0.083037,0.262949,0.092026,0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,1.145199,135000.0,45.931507,3.254795,0.070862,0.466757,0.311267,0.622246,0.219895,1
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,1.000000,67500.0,52.180822,0.616438,0.011814,0.642739,0.555912,0.729567,0.122792,1
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,1.052803,67500.0,52.068493,8.326027,0.159905,0.650442,0.650442,0.650442,NaN,2
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,1.000000,121500.0,54.608219,8.323288,0.152418,0.322738,0.322738,0.322738,NaN,2


## Bureau và Bureau_balance

In [7]:
#bureau: mỗi dòng là 1 khoản vay của khách hàng tại một tổ chức tín dụng khác Home Credit
bureau = pd.read_csv("../data/bureau.csv")
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [8]:
bureau[["DAYS_CREDIT","DAYS_ENDDATE_FACT","DAYS_CREDIT_UPDATE"]] = -1* bureau[["DAYS_CREDIT","DAYS_ENDDATE_FACT","DAYS_CREDIT_UPDATE"]]
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,497,0,-153.0,153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,131,NaN
1,215354,5714463,Active,currency 1,208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,20,NaN
2,215354,5714464,Active,currency 1,203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,16,NaN
3,215354,5714465,Active,currency 1,203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,16,NaN
4,215354,5714466,Active,currency 1,629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,21,NaN


In [9]:
bureau.describe()

,SK_ID_CURR,SK_ID_BUREAU,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
count,1.716428e+06,1.716428e+06,1.716428e+06,1.716428e+06,1.610875e+06,1.082775e+06,5.919400e+05,1.716428e+06,1.716415e+06,1.458759e+06,1.124648e+06,1.716428e+06,1.716428e+06,4.896370e+05
mean,2.782149e+05,5.924434e+06,1.142108e+03,8.181666e-01,5.105174e+02,1.017437e+03,3.825418e+03,6.410406e-03,3.549946e+05,1.370851e+05,6.229515e+03,3.791276e+01,5.937483e+02,1.571276e+04
std,1.029386e+05,5.322657e+05,7.951649e+02,3.654443e+01,4.994220e+03,7.140106e+02,2.060316e+05,9.622391e-02,1.149811e+06,6.774011e+05,4.503203e+04,5.937650e+03,7.207473e+02,3.258269e+05
min,1.000010e+05,5.000000e+06,0.000000e+00,0.000000e+00,-4.206000e+04,-0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-4.705600e+06,-5.864061e+05,0.000000e+00,-3.720000e+02,0.000000e+00
25%,1.888668e+05,5.463954e+06,4.740000e+02,0.000000e+00,-1.138000e+03,4.250000e+02,0.000000e+00,0.000000e+00,5.130000e+04,0.000000e+00,0.000000e+00,0.000000e+00,3.300000e+01,0.000000e+00
50%,2.780550e+05,5.926304e+06,9.870000e+02,0.000000e+00,-3.300000e+02,8.970000e+02,0.000000e+00,0.000000e+00,1.255185e+05,0.000000e+00,0.000000e+00,0.000000e+00,3.950000e+02,0.000000e+00
75%,3.674260e+05,6.385681e+06,1.666000e+03,0.000000e+00,4.740000e+02,1.489000e+03,0.000000e+00,0.000000e+00,3.150000e+05,4.015350e+04,0.000000e+00,0.000000e+00,9.080000e+02,1.350000e+04
max,4.562550e+05,6.843457e+06,2.922000e+03,2.792000e+03,3.119900e+04,4.202300e+04,1.159872e+08,9.000000e+00,5.850000e+08,1.701000e+08,4.705600e+06,3.756681e+06,4.194700e+04,1.184534e+08


In [10]:
bureau.isnull().mean()

SK_ID_CURR                0.000000
SK_ID_BUREAU              0.000000
CREDIT_ACTIVE             0.000000
CREDIT_CURRENCY           0.000000
DAYS_CREDIT               0.000000
CREDIT_DAY_OVERDUE        0.000000
DAYS_CREDIT_ENDDATE       0.061496
DAYS_ENDDATE_FACT         0.369170
AMT_CREDIT_MAX_OVERDUE    0.655133
CNT_CREDIT_PROLONG        0.000000
AMT_CREDIT_SUM            0.000008
AMT_CREDIT_SUM_DEBT       0.150119
AMT_CREDIT_SUM_LIMIT      0.344774
AMT_CREDIT_SUM_OVERDUE    0.000000
CREDIT_TYPE               0.000000
DAYS_CREDIT_UPDATE        0.000000
AMT_ANNUITY               0.714735
dtype: float64

In [11]:
bureau_bal = pd.read_csv("../data/bureau_balance.csv")
bureau_bal.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [12]:
bb = bureau_bal.copy()
status_map = {
    'X': 0,
    'C': 0,
    '0': 0,
    '1': 1,
    '2': 2,
    '3': 3,
    '4': 4,
    '5': 5
}

bb["DPD"] = bb["STATUS"].map(status_map).astype(int)

# Flag bad month
bb["BAD_MONTH"] = (bb["DPD"] > 0).astype(int)

bb["DPD30"] = (bb["DPD"] >= 2).astype(int)
bb["DPD60"] = (bb["DPD"] >= 3).astype(int)
bb["DPD90"] = (bb["DPD"] >= 4).astype(int)

# Khoản vay đang active
bb["ACTIVE_MONTH"] = (~bb["STATUS"].isin(["C", "X"])).astype(int)

def bureau_balance_window_feat(df, window):
    temp = df[df['MONTHS_BALANCE'] >= (-window +1)].copy()
    
    feature = (
        temp.groupby("SK_ID_BUREAU")
            .agg(
                max_dpd=("DPD", "max"),

                bad_months=("BAD_MONTH", "sum"),

                months_observed=("MONTHS_BALANCE", "count"),

                months_active=("ACTIVE_MONTH", "sum"),

                dpd30_months=("DPD30", "sum"),

                dpd60_months=("DPD60", "sum"),

                dpd90_months=("DPD90", "sum")
            )
            .reset_index()
    )
    
    #ratio of bad month
    feature['bad_ratio'] = feature['bad_months'] / feature['months_observed']
    
    feature = feature.rename(columns={
        "max_dpd": f"max_dpd_{window}m",
        "bad_months": f"bad_months_{window}m",
        "months_observed": f"months_observed_{window}m",
        "months_active": f"months_active_{window}m",
        "dpd30_months": f"dpd30_months_{window}m",
        "dpd60_months": f"dpd60_months_{window}m",
        "dpd90_months": f"dpd90_months_{window}m",
        "bad_ratio": f"bad_ratio_{window}m"
    })

    
    return feature

    
    

In [13]:
bb6 = bureau_balance_window_feat(bb, 6)

bb12 = bureau_balance_window_feat(bb, 12)

bb24 = bureau_balance_window_feat(bb, 24)

In [14]:
bureau_balance_feature = (
    bb6
    .merge(bb12, on="SK_ID_BUREAU", how="outer")
    .merge(bb24, on="SK_ID_BUREAU", how="outer")
)
bureau_balance_feature.head()

,SK_ID_BUREAU,max_dpd_6m,bad_months_6m,months_observed_6m,months_active_6m,dpd30_months_6m,dpd60_months_6m,dpd90_months_6m,bad_ratio_6m,max_dpd_12m,...,dpd90_months_12m,bad_ratio_12m,max_dpd_24m,bad_months_24m,months_observed_24m,months_active_24m,dpd30_months_24m,dpd60_months_24m,dpd90_months_24m,bad_ratio_24m
0,5001709,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,24,0,0,0,0,0.0
1,5001710,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,24,0,0,0,0,0.0
2,5001711,0.0,0.0,4.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,4,3,0,0,0,0.0
3,5001712,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,19,10,0,0,0,0.0
4,5001713,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,22,0,0,0,0,0.0


In [15]:
bureau_enriched = bureau.merge(
    bureau_balance_feature,
    on="SK_ID_BUREAU",
    how="left"
)
bureau_enriched.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,dpd90_months_12m,bad_ratio_12m,max_dpd_24m,bad_months_24m,months_observed_24m,months_active_24m,dpd30_months_24m,dpd60_months_24m,dpd90_months_24m,bad_ratio_24m
0,215354,5714462,Closed,currency 1,497,0,-153.0,153.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,208,0,1075.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,203,0,528.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,203,0,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,629,0,1197.0,NaN,77674.5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# =====================================
# Aggregate toàn bộ lịch sử theo SK_ID_BUREAU
# =====================================

bb_all = (
    bb.groupby("SK_ID_BUREAU")
      .agg(
          max_dpd_all=("DPD", "max"),
          bad_months_all=("BAD_MONTH", "sum"),
          months_observed_all=("MONTHS_BALANCE", "count"),
          months_active_all=("ACTIVE_MONTH", "sum"),
          dpd30_months_all=("DPD30", "sum"),
          dpd60_months_all=("DPD60", "sum"),
          dpd90_months_all=("DPD90", "sum")
      )
      .reset_index()
)

# Tỷ lệ tháng quá hạn trên toàn bộ lịch sử
bb_all["bad_ratio_all"] = (
    bb_all["bad_months_all"] /
    bb_all["months_observed_all"]
).round(4)

bureau_enriched_new = bureau_enriched.merge(
    bb_all,
    on="SK_ID_BUREAU",
    how="left"
)
bureau_enriched_new.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,dpd90_months_24m,bad_ratio_24m,max_dpd_all,bad_months_all,months_observed_all,months_active_all,dpd30_months_all,dpd60_months_all,dpd90_months_all,bad_ratio_all
0,215354,5714462,Closed,currency 1,497,0,-153.0,153.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,208,0,1075.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,203,0,528.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,203,0,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,629,0,1197.0,NaN,77674.5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
def bureau_features(df):

    temp = df.copy()

    temp["ACTIVE_LOAN"] = (temp["CREDIT_ACTIVE"] == "Active").astype(int)

    temp["CLOSED_LOAN"] = (temp["CREDIT_ACTIVE"] == "Closed").astype(int)

    temp["OVERDUE_LOAN"] = (temp["AMT_CREDIT_SUM_OVERDUE"] > 0).astype(int)

    temp["DPD30_6M_LOAN"] = (temp["max_dpd_6m"] >= 2).astype(int)

    temp["DPD30_12M_LOAN"] = (temp["max_dpd_12m"] >= 2).astype(int)

    temp["DPD30_24M_LOAN"] = (temp["max_dpd_24m"] >= 2).astype(int)

    feature = (
        temp.groupby("SK_ID_CURR").agg(
            
            # bad_ratio
            mean_bad_ratio_6m=("bad_ratio_6m","mean"),
            mean_bad_ratio_12m=("bad_ratio_12m","mean"),
            mean_bad_ratio_24m=("bad_ratio_24m","mean"),
            mean_bad_ratio_all=("bad_ratio_all","mean"),

            # Loan portfolio
            total_loans=("SK_ID_BUREAU", "count"),
            active_loans=("ACTIVE_LOAN", "sum"),
            closed_loans=("CLOSED_LOAN", "sum"),

            # Exposure
            total_credit=("AMT_CREDIT_SUM", "sum"),
            total_debt=("AMT_CREDIT_SUM_DEBT", "sum"),
            total_overdue=("AMT_CREDIT_SUM_OVERDUE", "sum"),

            overdue_loans=("OVERDUE_LOAN", "sum"),

            total_prolong=("CNT_CREDIT_PROLONG", "sum"),

            # Credit history
            oldest_credit_days=("DAYS_CREDIT", "min"),

            latest_enddate=("DAYS_CREDIT_ENDDATE", "max"),

            # Delinquency severity
            worst_dpd_6m=("max_dpd_6m", "max"),
            worst_dpd_12m=("max_dpd_12m", "max"),
            worst_dpd_24m=("max_dpd_24m", "max"),
            worst_dpd_all=("max_dpd_all", "max"),

            # Delinquency frequency
            bad_months_6m=("bad_months_6m", "sum"),
            bad_months_12m=("bad_months_12m", "sum"),
            bad_months_24m=("bad_months_24m", "sum"),
            bad_months_all=("bad_months_all", "sum"),

            # Number of delinquent loans
            dpd30_loans_6m=("DPD30_6M_LOAN", "sum"),
            dpd30_loans_12m=("DPD30_12M_LOAN", "sum"),
            dpd30_loans_24m=("DPD30_24M_LOAN", "sum")

        ).reset_index()
    )

    feature["active_ratio"] = (
        feature["active_loans"] /
        feature["total_loans"]
    ).round(4)

    feature["credit_utilization"] = (
        feature["total_debt"] /
        feature["total_credit"]
    )

    feature["credit_history_years"] = (
        feature["oldest_credit_days"] / 365
    ).round(2)

    return feature

In [18]:
bureau_feature = bureau_features(bureau_enriched_new)
bureau_feature.head()

,SK_ID_CURR,mean_bad_ratio_6m,mean_bad_ratio_12m,mean_bad_ratio_24m,mean_bad_ratio_all,total_loans,active_loans,closed_loans,total_credit,total_debt,...,bad_months_6m,bad_months_12m,bad_months_24m,bad_months_all,dpd30_loans_6m,dpd30_loans_12m,dpd30_loans_24m,active_ratio,credit_utilization,credit_history_years
0,100001,0.02381,0.011905,0.007519,0.007514,7,3,4,1453365.000,596686.5,...,1.0,1.0,1.0,1.0,0,0,0,0.4286,0.410555,0.13
1,100002,0.00000,0.000000,0.249206,0.255688,8,2,6,865055.565,245781.0,...,0.0,0.0,8.0,27.0,0,0,0,0.2500,0.284122,0.28
2,100003,NaN,NaN,NaN,NaN,4,1,3,1017400.500,0.0,...,0.0,0.0,0.0,0.0,0,0,0,0.2500,0.000000,1.66
3,100004,NaN,NaN,NaN,NaN,2,0,2,189037.800,0.0,...,0.0,0.0,0.0,0.0,0,0,0,0.0000,0.000000,1.12
4,100005,0.00000,0.000000,0.000000,0.000000,3,2,1,657126.000,568408.5,...,0.0,0.0,0.0,0.0,0,0,0,0.6667,0.864992,0.17


In [19]:
bureau_feature.isnull().mean().sort_values(ascending = True)

SK_ID_CURR              0.000000
closed_loans            0.000000
active_loans            0.000000
total_loans             0.000000
overdue_loans           0.000000
total_overdue           0.000000
total_debt              0.000000
total_credit            0.000000
total_prolong           0.000000
oldest_credit_days      0.000000
active_ratio            0.000000
dpd30_loans_6m          0.000000
bad_months_12m          0.000000
bad_months_24m          0.000000
bad_months_all          0.000000
bad_months_6m           0.000000
credit_history_years    0.000000
dpd30_loans_12m         0.000000
dpd30_loans_24m         0.000000
credit_utilization      0.003960
latest_enddate          0.008453
mean_bad_ratio_all      0.560049
worst_dpd_all           0.560049
mean_bad_ratio_24m      0.564335
worst_dpd_24m           0.564335
worst_dpd_12m           0.567543
mean_bad_ratio_12m      0.567543
mean_bad_ratio_6m       0.570401
worst_dpd_6m            0.570401
dtype: float64

## Previous_application

### pos_cash_balance

POS_CASH = Point of Sale + Cash Loan

Đây là lịch sử thanh toán hàng tháng của các khoản vay trả góp tại điểm bán (mua điện thoại, tivi, laptop trả góp...) hoặc các khoản cash loan trước đây của khách hàng tại Home Credit.

previous_application.csv trả lời câu hỏi "Khách hàng đã từng xin vay gì tại Home Credit?", còn POS_CASH_balance.csv trả lời câu hỏi "Sau khi khoản vay đó được giải ngân, khách hàng đã trả nợ như thế nào theo từng tháng?"

theo dõi tình trạng của khoản vay được liên kết từ previous_application

In [20]:
pcb = pd.read_csv("../data/POS_CASH_balance.csv")
pcb.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [21]:
pcb.isnull().mean()

SK_ID_PREV               0.000000
SK_ID_CURR               0.000000
MONTHS_BALANCE           0.000000
CNT_INSTALMENT           0.002607
CNT_INSTALMENT_FUTURE    0.002608
NAME_CONTRACT_STATUS     0.000000
SK_DPD                   0.000000
SK_DPD_DEF               0.000000
dtype: float64

In [22]:
pcb["NAME_CONTRACT_STATUS"].value_counts(normalize = True)


NAME_CONTRACT_STATUS
Active                   9.149876e-01
Completed                7.447819e-02
Signed                   8.724815e-03
Demand                   7.064041e-04
Returned to the store    5.460258e-04
Approved                 4.916332e-04
Amortized debt           6.359136e-05
Canceled                 1.499796e-06
XNA                      1.999728e-07
Name: proportion, dtype: float64

In [23]:
def pcb_features(df):
    temp_1 = df.copy()
    
    temp_1["BAD_MONTH"] = (temp_1["SK_DPD"] > 0).astype(int)
    
    temp_1["DPD30"] = (temp_1["SK_DPD"] > 30).astype(int)
    temp_1["DPD60"] = (temp_1["SK_DPD"] > 60).astype(int)
    temp_1["DPD90"] = (temp_1["SK_DPD"] > 90).astype(int)
    
    #-------------- all history --------------------------------
    
    feature_1 = (
    temp_1.groupby("SK_ID_CURR").agg(
        worst_pos_dpd_all=("SK_DPD", "max"),
        bad_months_all=("BAD_MONTH", "sum"),
        months_observed_all=("MONTHS_BALANCE", "count"),
        n_pos_loans=("SK_ID_PREV", "nunique")
    )
    .reset_index()
)

    feature_1["bad_ratio_all"] = (feature_1["bad_months_all"] /feature_1["months_observed_all"])

    
    
    
    # -------------- last 12 months -------------------------------
    temp_2 = temp_1[temp_1["MONTHS_BALANCE"] >= -12].copy()
    
    feature_2 = (temp_2.groupby("SK_ID_CURR").agg(
        worst_pos_dpd_12m=("SK_DPD", "max"),

            bad_months_12m=("BAD_MONTH", "sum"),

            months_observed_12m=("MONTHS_BALANCE", "count"),

            dpd30_months_12m=("DPD30", "sum"),

            dpd60_months_12m=("DPD60", "sum"),

            dpd90_months_12m=("DPD90", "sum")
    ).reset_index()
                 )
    
    feature_2["bad_ratio_12m"] = (feature_2["bad_months_12m"] /feature_2["months_observed_12m"])
    
    # =========================
    # COMPLETED LOANS
    # =========================

    completed_feat = (
        temp_1[
            temp_1["NAME_CONTRACT_STATUS"] == "Completed"
        ]
        .groupby("SK_ID_CURR")
        .agg(
            completed_loans=("SK_ID_PREV", "nunique"),

            recent_completed_month=(
                "MONTHS_BALANCE",
                "max"
            )
        )
        .reset_index()
    )
    
     # =========================
    # ACTIVE LOANS
    # =========================

    active_feat = (
        temp_1[
            temp_1["NAME_CONTRACT_STATUS"] == "Active"
        ]
        .groupby("SK_ID_CURR")
        .agg(
            active_pos_loans=("SK_ID_PREV", "nunique")
        )
        .reset_index()
    )
    
    
    # =========================
    # CURRENT BURDEN
    # Snapshot gần nhất của mỗi loan
    # =========================

    latest_snapshot = (
        temp_1.sort_values(
            ["SK_ID_PREV", "MONTHS_BALANCE"],
            ascending=[True, False]
        )
        .groupby("SK_ID_PREV")
        .head(1)
    )

    burden_feat = (
        latest_snapshot.groupby("SK_ID_CURR")
        .agg(
            total_future_instalments=(
                "CNT_INSTALMENT_FUTURE",
                "sum"
            ),

            mean_future_instalments=(
                "CNT_INSTALMENT_FUTURE",
                "mean"
            ),

            max_future_instalments=(
                "CNT_INSTALMENT_FUTURE",
                "max"
            )
        )
        .reset_index()
    )
    
    pos_features = (
        feature_1
        .merge(feature_2,
               on="SK_ID_CURR",
               how="left")
        .merge(completed_feat,
               on="SK_ID_CURR",
               how="left")
        .merge(active_feat,
               on="SK_ID_CURR",
               how="left")
        .merge(burden_feat,
               on="SK_ID_CURR",
               how="left")
    )

    return pos_features


In [24]:
pcb_feature = pcb_features(pcb)
pcb_feature.head()

,SK_ID_CURR,worst_pos_dpd_all,bad_months_all,months_observed_all,n_pos_loans,bad_ratio_all,worst_pos_dpd_12m,bad_months_12m,months_observed_12m,dpd30_months_12m,dpd60_months_12m,dpd90_months_12m,bad_ratio_12m,completed_loans,recent_completed_month,active_pos_loans,total_future_instalments,mean_future_instalments,max_future_instalments
0,100001,7,1,9,2,0.111111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,-53.0,2.0,0.0,0.000000,0.0
1,100002,0,0,19,1,0.000000,0.0,0.0,12.0,0.0,0.0,0.0,0.0,NaN,NaN,1.0,6.0,6.000000,6.0
2,100003,0,0,28,3,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,-18.0,3.0,1.0,0.333333,1.0
3,100004,0,0,4,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-24.0,1.0,0.0,0.000000,0.0
4,100005,0,0,11,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-15.0,1.0,0.0,0.000000,0.0


In [25]:
pcb_feature.isnull().mean()

SK_ID_CURR                  0.000000
worst_pos_dpd_all           0.000000
bad_months_all              0.000000
months_observed_all         0.000000
n_pos_loans                 0.000000
bad_ratio_all               0.000000
worst_pos_dpd_12m           0.287669
bad_months_12m              0.287669
months_observed_12m         0.287669
dpd30_months_12m            0.287669
dpd60_months_12m            0.287669
dpd90_months_12m            0.287669
bad_ratio_12m               0.287669
completed_loans             0.109746
recent_completed_month      0.109746
active_pos_loans            0.000646
total_future_instalments    0.000000
mean_future_instalments     0.000083
max_future_instalments      0.000083
dtype: float64

### credit card balance
Bảng này chứa lịch sử sử dụng thẻ tín dụng hàng tháng của khách hàng tại Home Credit trước khi họ nộp đơn vay hiện tại.

Không phải mọi đơn trong previous_application.csv đều là khoản vay trả góp; một số đơn là đăng ký sản phẩm thẻ tín dụng (revolving credit). Khi khách hàng được cấp thẻ tín dụng, Home Credit không theo dõi bằng các kỳ trả góp cố định mà theo dõi tài khoản thẻ theo từng kỳ sao kê hàng tháng. credit_card_balance.csv ghi nhận toàn bộ hoạt động này: dư nợ cuối kỳ, hạn mức tín dụng, số tiền đã chi tiêu, số tiền rút tiền mặt, số tiền khách hàng thanh toán và số ngày quá hạn nếu có.

In [26]:
ccb = pd.read_csv("../data/credit_card_balance.csv")
ccb.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [27]:
ccb.sort_values(["SK_ID_CURR","MONTHS_BALANCE"], ascending = [True,False]).head(13)

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
584804,1489396,100006,-1,0.0,270000,NaN,0.0,NaN,NaN,0.0,...,0.0,0.0,NaN,0,NaN,NaN,0.0,Active,0,0
520387,1489396,100006,-2,0.0,270000,NaN,0.0,NaN,NaN,0.0,...,0.0,0.0,NaN,0,NaN,NaN,0.0,Active,0,0
1347528,1489396,100006,-3,0.0,270000,NaN,0.0,NaN,NaN,0.0,...,0.0,0.0,NaN,0,NaN,NaN,0.0,Active,0,0
1399895,1489396,100006,-4,0.0,270000,NaN,0.0,NaN,NaN,0.0,...,0.0,0.0,NaN,0,NaN,NaN,0.0,Active,0,0
655566,1489396,100006,-5,0.0,270000,NaN,0.0,NaN,NaN,0.0,...,0.0,0.0,NaN,0,NaN,NaN,0.0,Active,0,0
1636141,1489396,100006,-6,0.0,270000,NaN,0.0,NaN,NaN,0.0,...,0.0,0.0,NaN,0,NaN,NaN,0.0,Active,0,0
2739019,1843384,100011,-2,0.0,90000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0.0,0.0,33.0,Active,0,0
3496910,1843384,100011,-3,0.0,90000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0.0,0.0,33.0,Active,0,0
51047,1843384,100011,-4,0.0,90000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0.0,0.0,33.0,Active,0,0
2674883,1843384,100011,-5,0.0,90000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0.0,0.0,33.0,Active,0,0


In [28]:
def ccb_features(df):
    temp = df.copy()
    
    # số dư nợ so với hạn mức tín dụng của thẻ
    temp["UTILIZATION"] = np.where(temp["AMT_CREDIT_LIMIT_ACTUAL"] > 0, temp["AMT_BALANCE"] / temp["AMT_CREDIT_LIMIT_ACTUAL"], np.nan) 
    
    cc = (temp.groupby("SK_ID_CURR").agg(
        avg_util_all=("UTILIZATION", "mean"),

        total_drawings=("AMT_DRAWINGS_CURRENT", "sum"),
        avg_drawings=("AMT_DRAWINGS_CURRENT", "mean"),

        max_dpd_cc=("SK_DPD", "max"),
        avg_dpd_cc=("SK_DPD", "mean"),

        months_observed_cc=("MONTHS_BALANCE", "count"),

        num_cc_accounts=("SK_ID_PREV", "nunique")
    ).reset_index())
    
    # ----------- Recent 6 Months ------------------------------
    recent_6m = temp[temp["MONTHS_BALANCE"] >= -5]
    
    cc_recent_6m = (temp.groupby("SK_ID_CURR").agg(
        avg_util_6m = ("UTILIZATION", "mean"),
        mã_util_6m = ("UTILIZATION","max")
    ).reset_index())
    
    # ----------- UTILIZATION trend ---------------------------- avg util (0,-1,-2) - avg util (-3,-4,-5)
    recent_3m = (temp[temp["MONTHS_BALANCE"] >= -2].groupby("SK_ID_CURR")["UTILIZATION"].mean())

    previous_3m = (temp[(temp["MONTHS_BALANCE"] <= -3)& (temp["MONTHS_BALANCE"] >= -5)].groupby("SK_ID_CURR")["UTILIZATION"].mean())

    cc_trend = ( pd.concat([recent_3m.rename("recent_util"),previous_3m.rename("previous_util")], axis=1).reset_index())

    cc_trend["util_trend"] = (cc_trend["recent_util"]- cc_trend["previous_util"])

    cc_trend = cc_trend[["SK_ID_CURR", "util_trend"]]
    
    # ----------- lastest statement ----------------------------
    lastest = (temp.sort_values(["SK_ID_CURR", "MONTHS_BALANCE"], ascending = [True, False]).groupby("SK_ID_CURR").first().reset_index())
    
    lastest["payment_ratio_last"] = np.where(lastest["AMT_BALANCE"]> 0, lastest["AMT_PAYMENT_TOTAL_CURRENT"] / lastest["AMT_BALANCE"], np.nan)
    
    cc_lastest = lastest[["SK_ID_CURR", "payment_ratio_last"]]
    
    cc_feature = (cc.merge(cc_recent_6m, on = "SK_ID_CURR", how = 'left')
                  .merge(cc_trend, on = "SK_ID_CURR", how = 'left')
                  .merge(cc_lastest, on ="SK_ID_CURR", how = 'left'))
    
    return cc_feature



In [29]:
ccb_feature = ccb_features(ccb)
ccb_feature.head()

,SK_ID_CURR,avg_util_all,total_drawings,avg_drawings,max_dpd_cc,avg_dpd_cc,months_observed_cc,num_cc_accounts,avg_util_6m,mã_util_6m,util_trend,payment_ratio_last
0,100006,0.000000,0.0,0.000000,0,0.000000,6,1,0.000000,0.00000,0.0,NaN
1,100011,0.302678,180000.0,2432.432432,0,0.000000,74,1,0.302678,1.05000,0.0,NaN
2,100013,0.115301,571500.0,5953.125000,1,0.010417,96,1,0.115301,1.02489,0.0,NaN
3,100021,0.000000,0.0,0.000000,0,0.000000,17,1,0.000000,0.00000,0.0,NaN
4,100023,0.000000,0.0,0.000000,0,0.000000,8,1,0.000000,0.00000,NaN,NaN


### Installments Payments

lưu lịch sử thanh toán thực tế của từng kỳ trả góp đối với các khoản vay trước đây của khách hàng tại Home Credit.

Nếu POS_CASH_balance.csv cho biết trạng thái khoản vay theo từng tháng, thì installments_payments.csv đi sâu hơn đến từng kỳ thanh toán. Sau khi một khoản vay trong previous_application.csv được giải ngân, hệ thống sẽ tạo lịch trả nợ (repayment schedule). Mỗi kỳ sẽ có một ngày đến hạn và một số tiền phải trả. installments_payments.csv lưu lại khách hàng đã trả bao nhiêu tiền, vào ngày nào, có trả đúng hạn hay không, trả thiếu hay trả dư cho từng kỳ. Nói cách khác, bảng này là lịch sử giao dịch thanh toán thực tế của khoản vay

Nếu POS_CASH_balance.csv cho biết tháng đó khoản vay có bị quá hạn hay không, thì installments_payments.csv cho biết nguyên nhân dẫn đến điều đó. Đây là bảng ghi nhận từng giao dịch thanh toán thực tế: khách hàng phải trả bao nhiêu, đã trả bao nhiêu, trả vào ngày nào, trả thiếu hay trả dư. Chính vì vậy, bảng này phản ánh rất rõ hành vi thanh toán của khách hàng.

In [30]:
ip = pd.read_csv("../data/installments_payments.csv")
ip.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [31]:
ip.sort_values(["SK_ID_CURR","NUM_INSTALMENT_NUMBER"], ascending= [True,True]).head(10)

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
1478621,1369693,100001,1.0,1,-1709.0,-1715.0,3951.000,3951.000
1761012,1851984,100001,1.0,2,-2916.0,-2916.0,3982.050,3982.050
2568722,1369693,100001,1.0,2,-1679.0,-1715.0,3951.000,3951.000
3458712,1369693,100001,1.0,3,-1649.0,-1660.0,3951.000,3951.000
3774071,1851984,100001,1.0,3,-2886.0,-2875.0,3982.050,3982.050
2624024,1369693,100001,2.0,4,-1619.0,-1628.0,17397.900,17397.900
3435373,1851984,100001,1.0,4,-2856.0,-2856.0,3980.925,3980.925
2144879,1038818,100002,1.0,1,-565.0,-587.0,9251.775,9251.775
2163032,1038818,100002,1.0,2,-535.0,-562.0,9251.775,9251.775
1675768,1038818,100002,1.0,3,-505.0,-529.0,9251.775,9251.775


In [32]:
ip.isna().mean()

SK_ID_PREV                0.000000
SK_ID_CURR                0.000000
NUM_INSTALMENT_VERSION    0.000000
NUM_INSTALMENT_NUMBER     0.000000
DAYS_INSTALMENT           0.000000
DAYS_ENTRY_PAYMENT        0.000214
AMT_INSTALMENT            0.000000
AMT_PAYMENT               0.000214
dtype: float64

In [33]:
ip.describe()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
count,1.360540e+07,1.360540e+07,1.360540e+07,1.360540e+07,1.360540e+07,1.360250e+07,1.360540e+07,1.360250e+07
mean,1.903365e+06,2.784449e+05,8.566373e-01,1.887090e+01,-1.042270e+03,-1.051114e+03,1.705091e+04,1.723822e+04
std,5.362029e+05,1.027183e+05,1.035216e+00,2.666407e+01,8.009463e+02,8.005859e+02,5.057025e+04,5.473578e+04
min,1.000001e+06,1.000010e+05,0.000000e+00,1.000000e+00,-2.922000e+03,-4.921000e+03,0.000000e+00,0.000000e+00
25%,1.434191e+06,1.896390e+05,0.000000e+00,4.000000e+00,-1.654000e+03,-1.662000e+03,4.226085e+03,3.398265e+03
50%,1.896520e+06,2.786850e+05,1.000000e+00,8.000000e+00,-8.180000e+02,-8.270000e+02,8.884080e+03,8.125515e+03
75%,2.369094e+06,3.675300e+05,1.000000e+00,1.900000e+01,-3.610000e+02,-3.700000e+02,1.671021e+04,1.610842e+04
max,2.843499e+06,4.562550e+05,1.780000e+02,2.770000e+02,-1.000000e+00,-1.000000e+00,3.771488e+06,3.771488e+06


In [34]:
ip.shape

(13605401, 8)

In [35]:
def ip_features(df):
    temp = df.copy()
    
    # --------------- PAYMENT RATIO ---------------
    temp['PAYMENT_RATIO'] = np.where(temp["AMT_INSTALMENT"] > 0, temp["AMT_PAYMENT"] / temp['AMT_INSTALMENT'], np.nan)
    
    # --------------- UNDER PAYMENT ---------------
    temp["UNPAID"] = (temp["PAYMENT_RATIO"] < 0.95).astype(int)
    
    #---------------- Days late ---------------------------
    # ------------ positive: not late, negative: late------
    temp['DAYS_LATE'] = temp["DAYS_ENTRY_PAYMENT"] - temp['DAYS_INSTALMENT']
    temp["LATE_PAY"] = (temp["DAYS_LATE"] > 0).astype(int)
    temp["LATE_7_DAYS"] = (temp["DAYS_LATE"] > 7).astype(int)
    
    # -------------- agg history ----------------------------
    feature_history = (temp.groupby("SK_ID_CURR").agg(
        avg_payment_ratio = ("PAYMENT_RATIO", "mean"),
        min_payment_ratio = ("PAYMENT_RATIO", "min"),
        
        underpaid_count = ("UNPAID", "sum"),
        
        avg_day_late = ("DAYS_LATE","mean"),
        max_day_late = ("DAYS_LATE","max"),
        
        late_count = ("LATE_PAY", "sum"),
        late_count_7 = ("LATE_7_DAYS", "sum"),
        
        total_installment = ("SK_ID_PREV", "count")
        
    ).reset_index())
    
    # -------------- last 12 months ---------------------
    last_12_months = temp[temp["DAYS_INSTALMENT"] >= - 365]
    feature_last_12 = (last_12_months.groupby("SK_ID_CURR").agg(
        avg_payment_12m = ("PAYMENT_RATIO", "mean"),
        
        underpaid_count_12m = ("UNPAID", "sum"),
        
        avg_day_late_12m = ("DAYS_LATE","mean"),
        
        installment_12m = ("SK_ID_PREV","count" ) 
    ).reset_index()) 
    
    # ------------- Late payment rate -----------------
    feature_history["late_payment_rate"] = feature_history['late_count'] / feature_history["total_installment"]
    
    feature = (feature_history.merge(feature_last_12, on = "SK_ID_CURR", how = "left"))
    
    return feature

In [36]:
ip_feature = ip_features(ip)
ip_feature.head()

,SK_ID_CURR,avg_payment_ratio,min_payment_ratio,underpaid_count,avg_day_late,max_day_late,late_count,late_count_7,total_installment,late_payment_rate,avg_payment_12m,underpaid_count_12m,avg_day_late_12m,installment_12m
0,100001,1.0,1.0,0,-7.285714,11.0,1,1,7,0.142857,NaN,NaN,NaN,NaN
1,100002,1.0,1.0,0,-20.421053,-12.0,0,0,19,0.000000,1.0,0.0,-17.583333,12.0
2,100003,1.0,1.0,0,-7.160000,-1.0,0,0,25,0.000000,NaN,NaN,NaN,NaN
3,100004,1.0,1.0,0,-7.666667,-3.0,0,0,3,0.000000,NaN,NaN,NaN,NaN
4,100005,1.0,1.0,0,-23.555556,1.0,1,0,9,0.111111,NaN,NaN,NaN,NaN


In [37]:
ip_feature.isna().mean()

SK_ID_CURR             0.000000
avg_payment_ratio      0.000035
min_payment_ratio      0.000035
underpaid_count        0.000000
avg_day_late           0.000027
max_day_late           0.000027
late_count             0.000000
late_count_7           0.000000
total_installment      0.000000
late_payment_rate      0.000000
avg_payment_12m        0.255781
underpaid_count_12m    0.255681
avg_day_late_12m       0.255781
installment_12m        0.255681
dtype: float64

### PREVIOUS APPLICAITON

previous_application.csv lưu thông tin về mọi lần khách hàng từng nộp đơn xin một sản phẩm tín dụng tại Home Credit, bất kể đơn đó được duyệt, bị từ chối, bị hủy hay khách hàng không sử dụng khoản vay sau khi được duyệt.

In [38]:
pre_app = pd.read_csv("../data/previous_application.csv")
pre_app.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
pre_app.isna().mean().round(2)

SK_ID_PREV                     0.00
SK_ID_CURR                     0.00
NAME_CONTRACT_TYPE             0.00
AMT_ANNUITY                    0.22
AMT_APPLICATION                0.00
AMT_CREDIT                     0.00
AMT_DOWN_PAYMENT               0.54
AMT_GOODS_PRICE                0.23
WEEKDAY_APPR_PROCESS_START     0.00
HOUR_APPR_PROCESS_START        0.00
FLAG_LAST_APPL_PER_CONTRACT    0.00
NFLAG_LAST_APPL_IN_DAY         0.00
RATE_DOWN_PAYMENT              0.54
RATE_INTEREST_PRIMARY          1.00
RATE_INTEREST_PRIVILEGED       1.00
NAME_CASH_LOAN_PURPOSE         0.00
NAME_CONTRACT_STATUS           0.00
DAYS_DECISION                  0.00
NAME_PAYMENT_TYPE              0.00
CODE_REJECT_REASON             0.00
NAME_TYPE_SUITE                0.49
NAME_CLIENT_TYPE               0.00
NAME_GOODS_CATEGORY            0.00
NAME_PORTFOLIO                 0.00
NAME_PRODUCT_TYPE              0.00
CHANNEL_TYPE                   0.00
SELLERPLACE_AREA               0.00
NAME_SELLER_INDUSTRY        

In [40]:
def previous_application_features(df):

    temp = df.copy()

    # -----------------Credit / Application Ratio -----------------------
    temp["CREDIT_APP_RATIO"] = np.where(
        temp["AMT_APPLICATION"] > 0,
        temp["AMT_CREDIT"] / temp["AMT_APPLICATION"],
        np.nan
    )

    #------------------ Status Flags ---------------------------
    temp["APPROVED"] = (
        temp["NAME_CONTRACT_STATUS"] == "Approved"
    ).astype(int)

    temp["REFUSED"] = (
        temp["NAME_CONTRACT_STATUS"] == "Refused"
    ).astype(int)

    temp["CANCELED"] = (
        temp["NAME_CONTRACT_STATUS"] == "Canceled"
    ).astype(int)

    temp["RECENT_12M"] = (
        temp["DAYS_DECISION"] >= -365
    ).astype(int)

    temp["INSURED"] = (
        temp["NFLAG_INSURED_ON_APPROVAL"] == 1
    ).astype(int)

    #-------------- Basic Aggregate ------------------------------
    
    prev_basic = (
        temp.groupby("SK_ID_CURR")
        .agg(
            total_applications=("SK_ID_PREV", "count"),

            approved_count=("APPROVED", "sum"),
            refused_count=("REFUSED", "sum"),
            canceled_count=("CANCELED", "sum"),

            applications_12m=("RECENT_12M", "sum"),

            insured_count=("INSURED", "sum")
        )
        .reset_index()
    )

    # --------------------------- Approval Rate------------------------
    prev_basic["approval_rate"] = (
        prev_basic["approved_count"]
        / prev_basic["total_applications"]
    )

    # --------------------------- Approved Loans Only -----------------
    approved = temp[
        temp["NAME_CONTRACT_STATUS"] == "Approved"
    ]

    prev_approved = (
        approved.groupby("SK_ID_CURR")
        .agg(
            avg_application_amount=("AMT_APPLICATION", "mean"),

            avg_credit_application_ratio=(
                "CREDIT_APP_RATIO",
                "mean"
            )
        )
        .reset_index()
    )

    # --------------------------- Latest Application ----------------------------
    # --------------------------- max(DAYS_DECISION) = gần observation point nhất
    latest_application = (
        temp.sort_values(
            ["SK_ID_CURR", "DAYS_DECISION"],
            ascending=[True, False]
        )
        .groupby("SK_ID_CURR")
        .first()
        .reset_index()
    )

    latest_application = latest_application[
        [
            "SK_ID_CURR",
            "AMT_CREDIT",
            "AMT_ANNUITY",
            "DAYS_DECISION"
        ]
    ].rename(
        columns={
            "AMT_CREDIT": "latest_credit_amount",
            "AMT_ANNUITY": "latest_annuity",
            "DAYS_DECISION": "latest_days_decision"
        }
    )
    
    
    # ----------------------- Days Since Latest Application -------------------
    latest_application["days_since_last_application"] = (
        -latest_application["latest_days_decision"]
    )

    
    # ---------------------- Merge ---------------------------------------------
    prev_features = (
        prev_basic
        .merge(
            prev_approved,
            on="SK_ID_CURR",
            how="left"
        )
        .merge(
            latest_application,
            on="SK_ID_CURR",
            how="left"
        )
    )

    return prev_features

In [41]:
pre_app_feature = previous_application_features(pre_app)
pre_app_feature.head()

,SK_ID_CURR,total_applications,approved_count,refused_count,canceled_count,applications_12m,insured_count,approval_rate,avg_application_amount,avg_credit_application_ratio,latest_credit_amount,latest_annuity,latest_days_decision,days_since_last_application
0,100001,1,1,0,0,0,0,1.0,24835.5,0.957782,23787.0,3951.000,-1740,1740
1,100002,1,1,0,0,0,0,1.0,179055.0,1.000000,179055.0,9251.775,-606,606
2,100003,3,3,0,0,0,2,1.0,435436.5,1.057664,1035882.0,98356.995,-746,746
3,100004,1,1,0,0,0,0,1.0,24282.0,0.828021,20106.0,5357.250,-815,815
4,100005,2,1,0,1,1,0,0.5,44617.5,0.899950,0.0,4813.200,-315,315


In [42]:
pre_app_feature.isna().mean()

SK_ID_CURR                      0.000000
total_applications              0.000000
approved_count                  0.000000
refused_count                   0.000000
canceled_count                  0.000000
applications_12m                0.000000
insured_count                   0.000000
approval_rate                   0.000000
avg_application_amount          0.003420
avg_credit_application_ratio    0.006726
latest_credit_amount            0.000000
latest_annuity                  0.001417
latest_days_decision            0.000000
days_since_last_application     0.000000
dtype: float64

## Merge

In [49]:
all_feature_train = (train_feature.merge(bureau_feature,on = "SK_ID_CURR", how = "left")
                     .merge(pcb_feature,on="SK_ID_CURR",how = "left")
                     .merge(ccb_feature,on = "SK_ID_CURR",how = "left")
                     .merge(ip_feature,on = "SK_ID_CURR",how = "left")
                     .merge(pre_app_feature,on = "SK_ID_CURR",how="left"))
all_feature_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,canceled_count,applications_12m,insured_count,approval_rate,avg_application_amount,avg_credit_application_ratio,latest_credit_amount,latest_annuity,latest_days_decision,days_since_last_application
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,1.000000,179055.000,1.000000,179055.0,9251.775,-606.0,606.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,2.0,1.000000,435436.500,1.057664,1035882.0,98356.995,-746.0,746.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,1.000000,24282.000,0.828021,20106.0,5357.250,-815.0,815.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,3.0,7.0,0.0,0.555556,352265.868,0.951861,675000.0,24246.000,-181.0,181.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,3.0,1.000000,150530.250,1.046356,274288.5,16037.640,-374.0,374.0


In [55]:
all_feature_train["TARGET"].shape


(307511,)

In [ ]:
all_feature_train.to_csv(
    "all_feature_train.csv",
    index=False,
    encoding="utf-8-sig"
)